![interpreto_banner](../assets/img/interpreto_banner.png)

# Generation Concept-based Explanation Tutorial

Welcome to this tutorial, our will be to obtain concept-based explanations starting from the beginning.

We will start with a minimal example to illustrate the pipeline.

1. [⏩ **Minimal** example: Top-k tokens for neurons](#minimal)

Then, in practice, there are four key steps for concepts based explanations:

0. [➗ **Split** your model in two parts](#split)
2. [🚦 Compute a dataset of **activations**](#activations)
3. [🏋️‍♂️ **Fit** a concept model on activations](#fit)
4. [🏷️ **Interpret** the concept dimensions](#interpret)

On which we add two bonus steps present in most papers:

5. [📍 **Local** concept analysis](#locally)
6. [⚖️ **Evaluate** concept-based explanations](#evaluate)

*Author: Antonin Poché*

## 0. ➗ **Split** your model in two parts <a class="anchor" id="split"></a>

Let's take a gpt2 for both the minimal example and the detailed pipeline. But you can naturally use larger models.

Here we split at the 11 / 12 layers. But you can specify the module path to split at.

To split the model, we use the [`interpreto.ModelWithSplitPoints`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/) which wraps around the `transformers` model and allows the computation of activations at the specified `split_points`.

> ➡️ **Note**
>
> Interpreto's splitting based on [`nnsight`](https://github.com/ndif-team/nnsight), so depending on your use case, you might want to use `nnsight` directly.

In [1]:
from transformers import AutoModelForCausalLM

from interpreto import ModelWithSplitPoints

# 1. load and split the the GPT2 model
mwsp = ModelWithSplitPoints(
    "gpt2",
    split_points=[10],           # split at the 11th layer
    automodel=AutoModelForCausalLM,
    device_map="cuda",
    batch_size=64,
)

## 1. ⏩ **Minimal** example: Top-k tokens for neurons <a class="anchor" id="minimal"></a>

In [2]:
from interpreto.concepts import NeuronsAsConcepts, TopKInputs

# 2. No dataset of activation is needed as we consider the latent space as the concept space

# 3. No training neither
# Use `NeuronsAsConcepts` to use the concept-based pipeline with neurons
concept_explainer = NeuronsAsConcepts(mwsp)

# 4. Use `TopKInputs` to get the top-k tokens that maximally activate each neuron
method = TopKInputs(
    concept_explainer=concept_explainer,
    use_vocab=True,             # use the vocabulary of the model and test all tokens (50257 with GPT2)
    k=10,                       # get the top 10 tokens for each neuron
)
topk_tokens = method.interpret(
    concepts_indices=list(range(5)),     # interpret the five first neurons
)

# show some neurons' interpretations
for concept_idx, tokens in topk_tokens.items():
    print(f"Concept {concept_idx}: {list(tokens.keys())}")

del concept_explainer, method, topk_tokens

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Concept 0: ['ItemTracker', 'Ġinterf', 'Ġcontrace', 'ĠactionGroup', 'ĠSho', 'Ġupro', 'Ġadversely', 'ĠPwr', 'Ġbapt', 'ĠDis']
Concept 1: ['Ġwatched', 'Ġinfring', 'Ġhostages', 'Ġchoke', 'Ġstared', 'Ġadorned', 'Ġbelieves', 'Ġdiscovers', 'Ġcontraceptives', 'Ġwatches']
Concept 2: ['ĠPeb', 'Kenn', 'ĠAmbro', 'ĠMillenn', 'ĠFaul', 'Fif', 'ĠFlore', 'ĠKod', 'ĠDAR', 'ĠKenn']
Concept 3: ['Ġ(@', 'ĠCould', 'ĠShould', '"></', 'ĠView', 'ĠWould', 'Ġcoron', 'Would', 'ĠguiName', '.</']
Concept 4: ['İĭ', 'Ġinval', '################', 'ĠORIG', 'oslov', 'ÃįÃį', 'DIR', 'Ð´', 'DERR', 'âĹ¼']


> ❓ **The concepts are not interpretable?**
>
> Well, this is surely due to superposition, this is why we use dictionary learning.
>
> Let's explore and solve this problem in the next sections.

## 2. 🚦 Compute a datasets of **activations** <a class="anchor" id="activations"></a>

We will use the [`IMDB`](https://huggingface.co/datasets/stanfordnlp/imdb) dataset to build a dataset of activations.

> ⚠️ **Warning**
>
> The dataset used has a considerable impact on the concept-space obtained. Hence, in practice, we recommend to use a subset of the model training set.
>
> The concept model we train be it SAEs or others, find patterns in the dataset of activations, which explains the dependence of the concepts found on the activations dataset.

> ➡️ **Note**
>
> The best dataset for SAEs would be one where each token is only seen once. But this is harder in practice, and such pipeline is not covered in this tutorial.

[`interpreto.ModelWithSplitPoints.get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/#interpreto.ModelWithSplitPoints.get_activations)

[`interpreto.ModelWithSplitPoints.activation_granularities`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/#interpreto.model_wrapping.model_with_split_points.ActivationGranularity)

In [3]:
from datasets import load_dataset

# take the whole dataset, more samples leads to better results, but some methods do not support too big datasets
imdb = load_dataset("stanfordnlp/imdb")["train"]["text"][:500]  # TODO: increase

# ignore special tokens activations
TOKEN = ModelWithSplitPoints.activation_granularities.TOKEN

# compute the activations of the whole IMDB dataset
# activations are flattened between the n_sample and seq_len dimensions
# which leads us to more then 6 million tokens
# (n * l, d)
activations_dict = mwsp.get_activations(
    inputs=imdb,
    activation_granularity=TOKEN,
    tqdm_bar=True,
)
# it is possible to compute activations for several split points
# hence, we need to extract the activations for the split point we are interested in
activations = mwsp.get_split_activations(activations_dict)

print(f"{activations.shape = }")

Computing activations: 100%|██████████| 8/8 [00:08<00:00,  1.05s/batch]

torch.Size([140508, 768])


## 3. 🏋️‍♂️ **Fit** a concept model on activations <a class="anchor" id="fit"></a>

Now we can fit a concept model on the activations. They exist more or less complex concept models. Here we use an SAE, so it is quite complex and has a lot more parameters than a simple concept model.

In particular, we use [`interpreto.concepts.BatchTopK`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/sae/#interpreto.concepts.methods.BatchTopKConcepts).

The `concept_explainer` wraps around both `mwsp`, the model wrapper, and the `concept_model`.

> ➡️ **Note**
> 
> Most of our concept models and all the SAEs implementations depend on the [Overcomplete](https://kempnerinstitute.github.io/overcomplete/) library. It is a library that provides a lot of concept models and optimization methods for concept extraction. So if you want to go deeper on these, we suggest digging there.

In [4]:
import torch

from interpreto.concepts.methods.overcomplete import BatchTopKSAEConcepts, DeadNeuronsReanimationLoss

top_k_individual = 5
concept_model_batch_size = 4096
epochs = 20

# instantiate the concept explainer with the splitted model
concept_explainer = BatchTopKSAEConcepts(
    mwsp, nb_concepts=500, device=mwsp.device,
    top_k=top_k_individual*concept_model_batch_size,
)

# train the SAE on the activations
log = concept_explainer.fit(
    activations=activations,
    criterion=DeadNeuronsReanimationLoss,  # set an MSE loss with dead neurons reanimation
    optimizer_class=torch.optim.Adam,
    scheduler_class=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_kwargs={"T_max": epochs, "eta_min": 1e-6},
    lr=1e-3,
    nb_epochs=epochs,
    batch_size=concept_model_batch_size,
    monitoring=1,
)

Epoch[1/20], Loss: 67.9441, R2: 0.1308, L0: 5.3275, Dead Features: 0.0%, Time: 1.1677 seconds
Epoch[2/20], Loss: 47.1588, R2: 0.3963, L0: 5.3275, Dead Features: 44.8%, Time: 0.7140 seconds
Epoch[3/20], Loss: 35.1952, R2: 0.5513, L0: 5.3275, Dead Features: 46.6%, Time: 0.4938 seconds
Epoch[4/20], Loss: 30.4381, R2: 0.6119, L0: 5.3275, Dead Features: 56.6%, Time: 0.8833 seconds
Epoch[5/20], Loss: 28.3950, R2: 0.6364, L0: 5.3275, Dead Features: 62.6%, Time: 0.6873 seconds
Epoch[6/20], Loss: 26.4542, R2: 0.6613, L0: 5.3275, Dead Features: 53.2%, Time: 0.5006 seconds
Epoch[7/20], Loss: 25.1334, R2: 0.6781, L0: 5.3275, Dead Features: 38.0%, Time: 0.6774 seconds
Epoch[8/20], Loss: 24.2314, R2: 0.6898, L0: 5.3275, Dead Features: 22.2%, Time: 0.7227 seconds
Epoch[9/20], Loss: 23.5345, R2: 0.6987, L0: 5.3275, Dead Features: 25.4%, Time: 0.6568 seconds
Epoch[10/20], Loss: 22.9900, R2: 0.7057, L0: 5.3275, Dead Features: 25.4%, Time: 0.6819 seconds
Epoch[11/20], Loss: 22.5488, R2: 0.7114, L0: 5.327

## 4. 🏷️ **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

Once the concept directions are set, it is time to interpret them.

Here we use [`interpreto.concepts.LLMLabels`](https://for-sight-ai.github.io/interpreto/), it uses LLM to label the concepts based on examples activating the concept.

In [5]:
import os

from interpreto.concepts import LLMLabels
from interpreto.model_wrapping.llm_interface import OpenAILLM

# set the LLM interface used to generate labels based on the constructed prompts
llm_interface = OpenAILLM(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-4.1-nano")

llm_labels_method = LLMLabels(
    concept_explainer=concept_explainer,
    activation_granularity=TOKEN,  # we suggest to use the same as the activations
    llm_interface=llm_interface,
    k_examples=10,  # number of examples in each concept
    k_context=10,  # number of tokens before and after the maximally activating one to give context
)

# compute the labels for an arbitrary subset of the concepts
interpretations = llm_labels_method.interpret(
    inputs=imdb,
    latent_activations=activations,  # as the inputs and granularity are the same, we can reuse the activations
    concepts_indices=list(range(10)),  # we could put `"all"` but that would take lon and cost a bit through the API
)

for concept_id, label in interpretations.items():
    print(f"Concept {concept_id}: {label if label is not None else 'None'}")

Concept 0: Initial tokens are highly emphasized, often indicating context or focus points.
Concept 1: Short, context-independent tokens with high activation at boundaries.
Concept 2: Text emphasizes specific tokens within context for importance.
Concept 3: The text emphasizes specific tokens by placing them within delimiters, highlighting their significance.
Concept 4: Text alternates between capitalized and lowercase words.
Concept 5: Text features include embedded tokens that influence meaning or classification, often emphasized by their context or surrounding structure.
Concept 6: Text emphasizes specific words by highlighting or selection, often for emphasis or focus.
Concept 7: Initial tokens are emphasized with high activation, indicating emphasis or importance. These tokens are typically at sentence beginnings, serving as expletives, pronouns, or adverbs. They often are single words or short phrases that set the tone or focus of the sentence. The remaining text shows low activat

> ➡️ **Note 1**
>
> The labels highly depend on the system prompt provided to the LLM interface.
>
> For labels formulation to better align with what you expect, you can set the `system_prompt` argument of the `LLMLabels`.
>
> Here is the default system prompt:

```python
SYSTEM_PROMPT_WITH_CONTEXT = """You are a meticulous AI researcher conducting an important investigation into patterns found in language.
Your task is to analyze text and provide an explanation that thoroughly encapsulates possible patterns found in it.
Guidelines:

You will be given a list of text examples on which special tokens are selected and between delimiters like <<this>>.
How important each token is for the behavior is listed after each example in parentheses, with importance from 0 to 10.

- Try to produce a concise final description. Simply describe the text features that are common in the examples, and what patterns you found.
- If the examples are uninformative, you don't need to mention them. Don't focus on giving examples of important tokens, but try to summarize the patterns found in the examples.
- Do not mention the marker tokens (<< >>) in your explanation.
- Do not make lists of possible explanations.
- Strike the balance between being concise and informative. From 3 to 7 words.
- Refrain from including uninformative elements like "patterns found include ...", "the examples show ...", or "text contains ...".
"""
```

> ➡️ **Note 2**
>
> Having our concepts and their interpretation is great. But what is really useful is to know:
> - When are each concept used? (see [next section](#locally))
> - If the concepts are pertinent. (see [final section](#evaluate))

## 5. 📍 **Local** concept analysis <a class="anchor" id="locally"></a>

To see the important concepts in a sample, there are two complementary steps:
- Find the most important concepts for the predicted tokens.
- Highlight the input tokens activating this concept.

### 5.0 Create a random sample and decompose it into tokens

In [21]:
from interpreto import Granularity

# create a sample
sample = ["Interpreto is magical! It means 'to interpret' in latin, and looks like an Harry Potter spell."]

# from text to ids back to tokens
sample_token_ids = mwsp.tokenizer(sample, return_tensors="pt")
sample_tokens = Granularity.get_decomposition(sample_token_ids, tokenizer=mwsp.tokenizer, granularity=TOKEN.value, return_text=True)[0]

print(f"The sample is: {sample[0]}\n\nIt is decomposed in {len(sample_tokens)} tokens:\n{sample_tokens}")

The sample is: Interpreto is magical! It means 'to interpret' in latin, and looks like an Harry Potter spell.

It is decomposed in 24 tokens:
['Inter', 'pret', 'o', ' is', ' magical', '!', ' It', ' means', " '", 'to', ' interpret', "'", ' in', ' lat', 'in', ',', ' and', ' looks', ' like', ' an', ' Harry', ' Potter', ' spell', '.']


### 5.1 Important concepts for text (with respect to the output)

Here we use the gradient of the concept-to-output function to estimate the importance of each concept for the outputs of the model.

[`interpreto.concepts.ConceptAutoEncoderExplainer.concept_output_gradient()`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/base/#interpreto.concepts.ConceptAutoEncoderExplainer.concept_output_gradient)

In [20]:
# (seq_len (out), seq_len (in), nb_concepts)
local_importances = concept_explainer.concept_output_gradient(
    inputs=sample,
    targets=None,  # all predicted tokens
    activation_granularity=TOKEN,  # same throughout the notebook
    concepts_x_gradients=True,  # based theoretically
    sample_target_normalization=False,  # for each output token, the sum of the absolute values of the importance is equal to 1
    batch_size=64,
)[0]  # only one sample

# we take the sum over the input sequence dimension, has we focus on the concept-output relationship
# (seq_len (out), nb_concepts)
local_importances = local_importances.abs().sum(dim=1)
print(f"{local_importances.shape = }")

local_importances.shape=torch.Size([24, 500])


### 5.2 Input tokens activating the concepts

In [ ]:
# compute the latent activations
# (seq_len, d_model)
local_activations = mwsp.get_split_activations(mwsp.get_activations(sample, TOKEN))

# and the concepts activations
# (seq_len, nb_concepts)
concepts_activations = concept_explainer.encode_activations(local_activations)
print(f"{concepts_activations.shape = }")

local_activations.shape=torch.Size([24, 768])
concepts_activations.shape=torch.Size([24, 500])


### 5.3 Visualization

In [ ]:
from interpreto.attributions import AttributionOutput
from interpreto.visualizations.concepts import ConceptHighlightVisualization

attr = AttributionOutput(elements=sample_tokens, attributions=attributions, model_task=ModelTask.GENERATION)

## 6. ⚖️ **Evaluate** concept-based explanations <a class="anchor" id="evaluate"></a>

### 6.1 Evaluate the concept-space from the [third part](#fit)

### 6.2 Evaluate the concepts-interpretations from the [fourth step](#important)